In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura plotagem gráfica inline para ambiente Jupyter
%matplotlib inline

# Decodificação intra-sujeito (*within-subject*) em ensaios reais

Estima a predição em ensaios retidos (*held-out*) de cada participante conhecido. Este diagnóstico
não estima o desempenho em um novo participante ou em uma nova sessão.

Dados: Nakanishi2015, NEMAR ``nm000118``, sujeitos 1–3, sessão 0, execução (*run*) 0:
aproximadamente 21.1 MB no primeiro download. Defina ``EEGDASH_CACHE_DIR`` para reutilizar o cache.
Esta versão processada já inclui filtragem, redução de taxa de amostragem (*downsampling*) e
tratamento de latência; não adicione outra correção de latência.
Consulte o [estudo original](https://doi.org/10.1371/journal.pone.0140703)
e o [lançamento NEMAR](https://nemar.org/dataset/nm000118).

Antes de executar, estude os tutoriais 02 e 11: uma janela possui um sinal, um alvo observado
e uma identidade de gravação. Instale o EEGDash com suas dependências Braindecode e scikit-learn;
este script não precisa de arquivos de saída anteriores. O resultado é uma pontuação diagnóstica
para cada pessoa conhecida. Escolha este protocolo quando houver ensaios de calibração disponíveis
dessa mesma pessoa.


## 1. Selecionar uma coorte pequena e explícita
Filtrar sujeitos, sessão e execução delimita o download. Cortar (*crop*) após abrir
uma gravação reduziria a computação, mas não o tamanho de download inicial.



In [ ]:
# Importa módulos de sistema operacional, funções parciais e manipulação de diretórios
import os
from functools import partial
from pathlib import Path

# Importa bibliotecas para plotagem, manipulação de arrays numéricos e DataFrames
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa gerador de janelas da Braindecode e modelo/métricas do scikit-learn
from braindecode.preprocessing import create_windows_from_events
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e extratores espectrais do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)

# Define o diretório de cache a partir de variável de ambiente ou valor padrão
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Lista de sujeitos selecionados para o experimento
subjects = ["1", "2", "3"]
# Inicializa e carrega os dados brutos de SSVEP dos sujeitos selecionados
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="nm000118",
    subject=subjects,
    session="0",
    run="0",
    task="ssvep",
    n_jobs=1,
)
# Assegura que há exatamente uma gravação correspondente a cada sujeito
assert len(dataset.datasets) == len(subjects), "Expected one recording per subject"
# Exibe os metadados descritivos das gravações carregadas
print(dataset.description[["subject", "session", "run"]])

## 2. Inspecionar anotações reais e verificar o contrato do sinal
Acessar ``raw`` faz o download dessa gravação. Os nomes das anotações identificam a frequência
do estímulo focado em Hz; eles fornecem todos os rótulos de classificação.
Todos os participantes devem ter a mesma ordem de canais e frequência de amostragem.



In [ ]:
# Acessa os dados brutos da primeira gravação para extrair parâmetros de referência
raw = dataset.datasets[0].raw
sfreq = raw.info["sfreq"]
channel_names = raw.ch_names
# Identifica as 12 frequências a partir das anotações ordenadas numericamente
class_names = sorted(set(raw.annotations.description), key=float)
# Cria dicionário de mapeamento nome_da_classe -> índice inteiro (0 a 11)
mapping = {name: index for index, name in enumerate(class_names)}
# Valida que há exatamente 12 classes de frequência
assert len(mapping) == 12, "Expected the twelve SSVEP stimulus frequencies"
# Verifica conformidade de canais, taxa de amostragem e anotações em cada gravação
for recording in dataset.datasets:
    recording_raw = recording.raw
    assert recording_raw.ch_names == channel_names
    assert recording_raw.info["sfreq"] == sfreq
    assert set(recording_raw.annotations.description) == set(mapping)
# Imprime informações dos canais, taxa e frequências identificadas
print(f"Channels: {channel_names}; sampling frequency: {sfreq} Hz")
print("Stimulus frequencies (Hz):", class_names)

## 3. Fazer uma janela de quatro segundos por ensaio anotado
Cada intervalo anotado dura 4.15 segundos. Mantenha seus primeiros quatro segundos e descarte o restante.
O tamanho e o passo (*stride*) explícitos evitam janelas sobrepostas ou a extensão da época
além da duração do evento gravado.
A 256 Hz, quatro segundos contêm 1.024 amostras. O array resultante tem eixos
(540 ensaios, 8 canais de EEG, 1.024 amostras), com dados expressos em volts.
Os metadados possuem uma linha por linha do array. ``target`` é um índice de classe, não uma
frequência em Hz; ``mapping`` é a conversão explícita entre eles.
A fonte contém 15 ensaios de cada uma das 12 frequências por pessoa.
Uma classe ausente seria uma falha no contrato de dados, não um motivo para rotular ensaios novamente.



In [ ]:
# Define o tamanho da janela em amostras (4 segundos * 256 Hz = 1024 amostras)
window_size = int(4 * sfreq)
# Cria as janelas a partir dos eventos descartando a sobra fracionária final de 0.15s
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Extrai os metadados associados a cada janela
metadata = windows.get_metadata()
# Garante que há apenas uma única janela para cada ensaio registrado
assert (metadata.i_window_in_trial == 0).all(), "Expected one window per trial"
# Confirma ausência de duplicatas em ensaios
assert not metadata.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Extrai array numpy de rótulos numéricos e grupos de participantes
y = metadata["target"].to_numpy(dtype=int)
groups = metadata["subject"].astype(str).to_numpy()
# Empilha os dados em uma matriz 3D (540 ensaios, 8 canais, 1024 amostras)
X = np.stack([window[0] for window in windows])
# Valida o formato dimensional e finitude numérica de X
assert X.shape == (len(metadata), len(channel_names), window_size)
assert set(groups) == set(subjects)
assert np.isfinite(X).all()
# Exibe tabela cruzada demonstrando que cada sujeito possui exatamente 15 ensaios por classe
print(pd.crosstab(groups, y, rownames=["subject"], colnames=["class"]))

## 4. Extrair características espectrais de cada janela
As respostas de SSVEP contêm energia na frequência do estímulo. Use o log da potência espectral
em torno de cada frequência de estímulo, mantendo todos os oito canais posteriores.
Essa transformação por janela não aprende nada de outros ensaios ou sujeitos.
O escalador abaixo, por outro lado, deve ser ajustado apenas nos sujeitos de treino.
O pré-processador espectral compartilhado do EEGDash calcula uma PSD de Welch com um segmento Hann
de quatro segundos e bins de 0.25 Hz. Cada banda estreita é centrada em uma frequência de estímulo
documentada; esses centros definem a tarefa, não preditores específicos de ensaios.
Manter oito canais resulta em 12 × 8 = 96 características. A versão do SSVEP já processada
não precisa de outra etapa de limpeza com EEGPrep ou correção visual de latência.

``spectral_bands_power`` soma os bins selecionados da PSD. Multiplicar pelo seu espaçamento
de 0.25 Hz converte V²/Hz em potência aproximada da banda em V². O logaritmo comprime
essa escala; o StandardScaler ainda se ajusta apenas aos participantes de treino.



In [ ]:
# Define bandas estreitas de ±0.125 Hz ao redor de cada frequência de estímulo anotada
bands = {
    f"hz_{name}": (float(name) - 0.125, float(name) + 0.125) for name in class_names
}
# Configura o extrator com Welch PSD de 4 segundos (nperseg=1024, resolução de 0.25 Hz)
spectral = FeatureExtractor(
    {"power": partial(spectral_bands_power, bands=bands)},
    preprocessor=partial(
        spectral_preprocessor,
        fs=sfreq,
        nperseg=window_size,
        noverlap=0,
        f_min=8,
        f_max=16,
    ),
)
# Executa a extração em lote para todas as janelas
feature_table = extract_features(
    windows, {"spectral": spectral}, batch_size=64, n_jobs=1
).to_dataframe()
# Assegura que o número de colunas é exatamente 12 bandas * 8 canais = 96 características
assert feature_table.shape == (len(y), len(class_names) * len(channel_names))
# Converte para potência em V² multiplicando pelo bin width (sfreq / window_size = 0.25 Hz) e aplica log natural com piso de 1e-30
features = np.log(np.maximum(feature_table.to_numpy() * sfreq / window_size, 1e-30))
# Garante finitude matemática de todas as características computadas
assert np.isfinite(features).all()

## 5. Reter ensaios completos dentro de cada participante
Exatamente uma janela representa cada ensaio, portanto dividir os índices de janela aqui
também divide os ensaios. Com janelas sobrepostas, agrupe pelo ensaio original.
Esta versão concatena os ensaios; uma divisão aleatória não faz nenhuma reivindicação cronológica.



In [ ]:
# Importa utilitário de divisão treino/teste estratificada do scikit-learn
from sklearn.model_selection import train_test_split

A fração de teste de 25% deixa 135 ensaios de treino e 45 de teste por sujeito.
A estratificação preserva a representação de todas as doze classes; com 15 ensaios por classe,
proporções exatas de classe nem sempre podem ser retidas após o arredondamento de inteiros.
A semente fixa fixa as atribuições para que mudanças no código possam ser comparadas
sem alterar silenciosamente os ensaios retidos.

O StandardScaler estima a média e a dispersão de cada característica usando os ensaios de treino.
A LogisticRegression então ajusta seu classificador padrão com regularização L2;
max_iter=1000 é um limite de otimização, não uma busca de hiperparâmetros. Se ocorrerem avisos
de convergência, inspecione as escalas das características e a convergência do solucionador antes
de interpretar a pontuação.



In [ ]:
# Lista para armazenar as métricas de cada sujeito
rows = []
# Avalia cada sujeito de forma independente (intra-sujeito)
for subject in subjects:
    # Obtém os índices dos ensaios pertencentes ao sujeito atual
    indices = np.flatnonzero(groups == subject)
    # Divide em 75% treino e 25% teste estratificado pelas 12 classes com semente 42
    train, test = train_test_split(
        indices, test_size=0.25, random_state=42, stratify=y[indices]
    )
    # Garante ausência de sobreposição entre treino e teste
    assert set(train).isdisjoint(test)
    # Garante que todas as 12 classes constam no treino e no teste
    assert set(y[train]) == set(y[test]) == set(mapping.values())
    # Constrói o pipeline com padronizador e regressão logística com até 1000 iterações
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    # Ajusta o modelo estritamente no treino desse sujeito específico
    model.fit(features[train], y[train])
    # Gera predições para os 45 ensaios de teste retidos do mesmo sujeito
    prediction = model.predict(features[test])
    # Registra o resultado da acurácia balanceada e o número de ensaios de teste
    rows.append(
        dict(
            subject=subject,
            balanced_accuracy=balanced_accuracy_score(y[test], prediction),
            n_test=len(test),
        )
    )
# Monta e exibe a tabela com as pontuações diagnósticas de cada sujeito
results = pd.DataFrame(rows)
print(results.to_string(index=False))

## 6. Plotar o desempenho real retido



In [ ]:
# Cria o gráfico de barras com a acurácia balanceada obtida para cada sujeito
results.plot.bar(x="subject", y="balanced_accuracy", legend=False)
# Traça linha horizontal indicando o nível de chance para 12 classes (1/12 ≈ 8.33%)
plt.axhline(1 / len(mapping), color="black", linestyle="--", label="Chance (1/12)")
# Configura rótulo vertical e limites do gráfico de 0 a 1
plt.ylabel("Held-out trial balanced accuracy")
plt.ylim(0, 1)
plt.legend()
# Exibe a figura
plt.show()

## 7. Interpretar o diagnóstico e escolher a próxima divisão
A acurácia balanceada calcula a média da sensibilidade (*recall*) entre as frequências de estímulo,
de modo que cada frequência contribui igualmente, mesmo que as contagens retidas variem.
A linha de chance é 1/12 para uma regra de predição uniforme de doze classes. Uma barra acima
dessa linha não é um teste de significância estatística, e cada barra depende de uma divisão
aleatória com apenas 45 ensaios de teste.

Para um experimento de tamanho de calibração, mantenha esses índices de teste fixos e reduza
apenas os ensaios de treino usando subconjuntos estratificados por classe. Para responder se o decodificador
funciona em uma pessoa não vista, use o tutorial 51; uma pontuação intra-sujeito não pode responder
a essa pergunta de implantação em produção.

